# Ask CIP v6: Ask picks on its own, with no AI API

Ask sends its AI a numbered list of candidate survey questions and takes its pick; the rules do the rest (period, group,
kind: right on every golden question). With Qwen3-32B's pick in the AI's place, Ask was right on 154 of 161 golden
questions (no OpenAI). v6 teaches a small model those picks, from exactly the lines the AI reads (waves, years, name,
wording, answers), so the server picks on its own CPU in a fraction of a second, with no API at all.

1. **The teacher** (Qwen3-32B, open, on this GPU) picks for ~11,000 practice questions, the golden set and 2,500 of the
   big exam; hard ones read again with thinking if that does better on the tests. `teacher-v6.json` downloads when done:
   keep it (uploading it again skips its plain picks, the long part; a thinking check that is missing or came back
   unreadable is read again).
2. **The student** (MiniLM, 22M or 33M dials) learns the picks; "none of these" is one of its choices.
3. **Shrunk to 8-bit**, checked with the server's own code, timed on one CPU thread; `ask-v6.zip` downloads: send it back (scored on the Mac by `scripts/ask_v6.py score`).

**Run:** Runtime → Change runtime type → **A100** → Runtime → Run all → pick `v6-train.json` and `v6-tests.json` (and
`teacher-v6.json` if you have it) from `Downloads/cip-colab`. About 2½-3 hours (~30-35 units); with the 14 November
`teacher-v6.json` uploaded, about 1½-2½ hours (only its thinking check is read again). Keep the laptop open.
No API key is used anywhere; the "HF_TOKEN" warning is harmless.

In [ ]:
# 1. Libraries (vLLM brings its own PyTorch, so it goes first), the GPU, the files
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "onnx", "onnxruntime"], check=True)
import json, os, random, shutil, time
import numpy as np, torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100, then Run all again"
MEMORY = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{MEMORY:.0f} GB")
if not (os.path.exists("v6-train.json") and os.path.exists("v6-tests.json")):
    from google.colab import files
    print("Pick v6-train.json and v6-tests.json (and teacher-v6.json if you have it) from Downloads/cip-colab (hold ⌘)")
    files.upload()
data, tests = json.load(open("v6-train.json")), json.load(open("v6-tests.json"))
print(len(data["items"]), "practice questions ·", len(tests["golden"]), "golden ·", len(tests["bank"]), "big exam")

In [ ]:
%%writefile teacher.py
# The teacher, in its own process so the GPU is free when it exits: Qwen3-32B (AWQ, 4-bit) on vLLM picks a candidate
# for every request, its reply held to Ask's own JSON schema. Hard requests (its pick is not the question the practice
# question was written for) are read again with thinking, if thinking does better on the tests. Saves as it goes.
import json, os, re, sys, time
from vllm import LLM, SamplingParams
data, tests = json.load(open("v6-train.json")), json.load(open("v6-tests.json"))
SYSTEM, SCHEMA = data["system"], data["schema"]
ENUMS = {k: v["enum"] for k, v in SCHEMA["properties"].items() if "enum" in v}
llm = LLM("Qwen/Qwen3-32B-AWQ", max_model_len=8192, gpu_memory_utilization=0.92, enable_prefix_caching=True)
tok = llm.get_tokenizer()

def held_to_schema(**kw):
    # vLLM renamed this option in 2025: the new name first, then the old, else no guidance (replies are checked below)
    try:
        from vllm.sampling_params import StructuredOutputsParams
        return SamplingParams(structured_outputs=StructuredOutputsParams(json=SCHEMA), **kw)
    except (ImportError, TypeError):
        pass
    try:
        from vllm.sampling_params import GuidedDecodingParams
        return SamplingParams(guided_decoding=GuidedDecodingParams(json=SCHEMA), **kw)
    except (ImportError, TypeError):
        print("vLLM without schema guidance: replies are checked instead")
        return SamplingParams(**kw)

def prompt(user, thinking):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=thinking)

def pick(text, n):
    # the candidate number it chose, None for "none of these", -1 for a reply that cannot be read
    text = text.split("</think>")[-1]
    found = re.search(r"\{.*\}", text, re.S)
    try:
        q = json.loads(found.group(0)).get("q") if found else -1
    except ValueError:
        return -1
    return q if q is None or (isinstance(q, int) and 0 <= q < n) else -1

def read(items, thinking, label):
    params = (SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=4096) if thinking
              else held_to_schema(temperature=0, max_tokens=160))
    out, started = [], time.time()
    for s in range(0, len(items), 2000):
        part = items[s:s + 2000]
        try:
            made = llm.generate([prompt(i["user"], thinking) for i in part], params, use_tqdm=False)
        except Exception as exc:   # a schema the guidance cannot hold: the same requests without it
            print(f"  guided reading failed ({type(exc).__name__}); reading without it")
            made = llm.generate([prompt(i["user"], thinking) for i in part], SamplingParams(temperature=0, max_tokens=160), use_tqdm=False)
        out += [pick(m.outputs[0].text, len(i["lines"])) for m, i in zip(made, part)]
        print(f"  {label}: {len(out)} of {len(items)} · {time.time() - started:.0f}s", flush=True)
    return out

def right(test, picks):
    # a proxy for Ask's score: the picked candidate's wording has a golden phrase, or is the big exam's answer key
    ok = 0
    for i, p in zip(tests[test], picks):
        if test == "golden":
            ok += (p is None) if i["want"] == "none" else (p is not None and p >= 0 and any(w in i["texts"][p].lower() for w in i["want"]))
        else:
            ok += p is not None and p >= 0 and i["keys"][p] == i["answer_key"]
    return ok

done = json.load(open("teacher-v6.json")) if os.path.exists("teacher-v6.json") else {}
for test in ("golden", "bank"):
    if test not in done:
        done[test] = read(tests[test], False, test)
        json.dump(done, open("teacher-v6.json", "w"))
    print(f"{test}: the teacher picks the right one for {right(test, done[test])} of {len(tests[test])}", flush=True)
if "train" not in done:
    done["train"] = read(data["items"], False, "practice questions")
    json.dump(done, open("teacher-v6.json", "w"))
if set(done.get("thinking golden", [-1])) == {-1}:   # not read yet, or every reply cut short (2,048 tokens on 14 Nov): read again
    done["thinking golden"], done["thinking bank"] = read(tests["golden"], True, "golden, thinking"), read(tests["bank"][:1000], True, "big exam (1,000), thinking")
    json.dump(done, open("teacher-v6.json", "w"))
gain = sum(right(t, done[f"thinking {t}"]) - right(t, done[t][:len(done[f"thinking {t}"])]) for t in ("golden", "bank"))   # the same questions
print(f"thinking on the tests: {gain:+d} right answers against without", flush=True)
if gain > 0 and "thinking train" not in done:
    hard = [n for n, (i, p) in enumerate(zip(data["items"], done["train"]))
            if p == -1 or i["source"] is None or p is None or i["keys"][p] != i["source"]][:5000]
    again = read([data["items"][n] for n in hard], True, "hard practice questions, thinking")
    done["thinking train"] = {str(n): p for n, p in zip(hard, again)}
    json.dump(done, open("teacher-v6.json", "w"))
print("teacher done", flush=True)


In [ ]:
# 2. The teacher (skipped when teacher-v6.json is complete). It needs an A100: Qwen3-32B is 19 GB before its working memory
done = json.load(open("teacher-v6.json")) if os.path.exists("teacher-v6.json") else {}
if "train" not in done or set(done.get("thinking golden", [-1])) == {-1}:   # the thinking check not read yet, or unreadable
    assert MEMORY > 30, "The teacher needs an A100 (40 GB). With teacher-v6.json uploaded, an L4 can run the rest."
    run = subprocess.Popen([sys.executable, "teacher.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in run.stdout:   # its progress, without vLLM's own chatter
        if not line.startswith(("INFO", "WARNING", "(", "[")) and line.strip():
            print(line, end="")
    run.wait()
    assert run.returncode == 0 and os.path.exists("teacher-v6.json"), "the teacher stopped: the lines above say why"
    from google.colab import files
    files.download("teacher-v6.json")   # keep it: a re-run that uploads it skips this step
teacher = json.load(open("teacher-v6.json"))
labels = list(teacher["train"])
for n, p in teacher.get("thinking train", {}).items():
    if p != -1:
        labels[int(n)] = p
print(f"labels: {sum(p is not None and p >= 0 for p in labels)} picks · {sum(p is None for p in labels)} 'none of these' · "
      f"{sum(p == -1 for p in labels)} unreadable (left out) · thinking used for {len(teacher.get('thinking train', {}))}")

In [ ]:
# 3. The student: for one question, a score per candidate line (the line the AI reads, without its number); "none of
# these" is a fixed score of 0, so a candidate is picked only when it scores above 0. Trained on the teacher's picks
from transformers import AutoModelForSequenceClassification, AutoTokenizer
DEVICE = "cuda"
def line_text(line):
    return line.split(". ", 1)[1] if ". " in line[:5] else line
examples = [(i["q"], [line_text(l) for l in i["lines"]], p) for i, p in zip(data["items"], labels) if p != -1]
random.Random(5).shuffle(examples)
held, train_ex = examples[:900], examples[900:]

def scores(model, tok, question, lines):
    enc = tok([question] * len(lines), lines, truncation=True, max_length=192, padding=True, return_tensors="pt").to(DEVICE)
    return model(**enc).logits.view(-1)

def choose(s):
    best = int(torch.argmax(s))
    return best if float(s[best]) > 0 else None

def agreement(model, tok, rows):
    model.eval()
    with torch.no_grad():
        return float(np.mean([choose(scores(model, tok, q, l)) == p for q, l, p in rows]))

def train(name, epochs=3, per_step=16, lr=3e-5):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * (len(train_ex) // per_step)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (steps // 10 + 1)) * max(0.0, 1 - s / steps))
    best, best_state = -1.0, None
    for epoch in range(epochs):
        model.train(); random.Random(epoch).shuffle(train_ex); started = time.time()
        for s in range(0, len(train_ex) - per_step + 1, per_step):
            loss = 0
            for q, lines, p in train_ex[s:s + per_step]:   # softmax over "none" (0) and every candidate
                logits = torch.cat([torch.zeros(1, device=DEVICE), scores(model, tok, q, lines)])
                loss = loss + torch.nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([0 if p is None else p + 1], device=DEVICE))
            (loss / per_step).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
        now = agreement(model, tok, held)
        print(f"  {name} epoch {epoch + 1}: agrees with the teacher on {now:.1%} of held-out questions · {time.time() - started:.0f}s")
        if now > best:
            best, best_state = now, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model.eval(), tok, best

students = {name: train(name) for name in ("cross-encoder/ms-marco-MiniLM-L6-v2", "cross-encoder/ms-marco-MiniLM-L12-v2")}

In [ ]:
# 4. Each student shrunk to the server's format (8-bit ONNX), read with the server's own code (api/ask_model.py read_pick),
# timed on one CPU thread (Colab's CPU is slower than the server's: both students go back, and the Mac times and scores
# them inside Ask with no AI: scripts/ask_v6.py score)
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
NAMES = ["input_ids", "attention_mask", "token_type_ids"]
options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1

def shrink(model, tok, folder):
    os.makedirs(folder, exist_ok=True)
    dummy = tok(["a question"], ["a survey question"], return_tensors="pt")
    torch.onnx.export(model.cpu().eval(), tuple(dummy[k] for k in NAMES), "full.onnx", input_names=NAMES, output_names=["logits"],
                      dynamic_axes={k: {0: "batch", 1: "tokens"} for k in NAMES} | {"logits": {0: "batch"}}, opset_version=17, dynamo=False)
    quantize_dynamic("full.onnx", f"{folder}/model_optimized.onnx", weight_type=QuantType.QInt8)
    tok.save_pretrained("tok_tmp"); shutil.copy("tok_tmp/tokenizer.json", f"{folder}/tokenizer.json")
    model.to(DEVICE)
    words = tokenizers.Tokenizer.from_file(f"{folder}/tokenizer.json"); words.enable_truncation(192); words.enable_padding(pad_id=0, pad_token="[PAD]")
    session = onnxruntime.InferenceSession(f"{folder}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
    def server_pick(question, lines):   # api/ask_model.py read_pick, line for line, then above 0.5 or "none"
        coded = words.encode_batch([(question, t) for t in lines]); ids = np.array([c.ids for c in coded], np.int64)
        feed = {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64), "token_type_ids": np.array([c.type_ids for c in coded], np.int64)}
        p = 1 / (1 + np.exp(-session.run(None, feed)[0].reshape(-1)))
        return int(np.argmax(p)) if p.max() > 0.5 else None
    return server_pick

def proxy(test, picks):   # as in teacher.py: the golden phrase or the big exam's key
    ok = 0
    for i, p in zip(tests[test], picks):
        if test == "golden":
            ok += (p is None) if i["want"] == "none" else (p is not None and p >= 0 and any(w in i["texts"][p].lower() for w in i["want"]))
        else:
            ok += p is not None and p >= 0 and i["keys"][p] == i["answer_key"]
    return ok

results, picks_by = {"teacher": {t: proxy(t, teacher[t]) for t in ("golden", "bank")}}, {}
for name, (model, tok, agree) in students.items():
    short = name.split("-")[-2]
    server_pick = shrink(model, tok, f"ask-v6/ask-reader-{short}")
    started = time.time()
    picks = {t: [server_pick(i["q"], [line_text(l) for l in i["lines"]]) for i in tests[t]] for t in ("golden", "bank")}
    ms = (time.time() - started) / sum(len(tests[t]) for t in ("golden", "bank")) * 1000
    same = float(np.mean([server_pick(q, l) == p for q, l, p in held[:300]]))
    results[short] = {"agrees with the teacher": agree, "8-bit agrees": same, "ms a question, one CPU thread": round(ms),
                      **{t: proxy(t, picks[t]) for t in ("golden", "bank")}}
    print(f"{short}: agrees with the teacher {agree:.1%} (8-bit {same:.1%}) · golden {results[short]['golden']} · big exam "
          f"{results[short]['bank']} (teacher {results['teacher']['golden']}, {results['teacher']['bank']}) · {ms:.0f} ms a question")
    picks_by[short] = picks

In [ ]:
# 5. The zip: both students (each a server models/ask-reader folder), their picks and the teacher's for the tests as Ask's AI
# replies (scored on the Mac with no AI), the teacher's labels and the numbers
import zipfile
def as_reply(p):
    return json.dumps({"q": p, "r": None, "a": [], "k": "share", "g": "auto", "c": "none", "p": "all"})
for short in picks_by:
    json.dump({"sure": 2.0, "length": 192, "reads": "api/ask_model.py describe_candidate, the AI's line without its number"}, open(f"ask-v6/ask-reader-{short}/calibration.json", "w"))
for who, picks in [(f"v6-{short}", p) for short, p in picks_by.items()] + [("teacher", {t: teacher[t] for t in ("golden", "bank")})]:
    for t in ("golden", "bank"):   # an unreadable teacher reply (-1) is left out: Ask treats it as a failed AI call
        json.dump({i["user"]: as_reply(p) for i, p in zip(tests[t], picks[t]) if p != -1}, open(f"ask-v6/replies-{who}-{t}.json", "w"))
json.dump(results, open("ask-v6/results.json", "w"), indent=1)
shutil.copy("teacher-v6.json", "ask-v6/")
with zipfile.ZipFile("ask-v6.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, names in os.walk("ask-v6"):
        for n in names:
            z.write(os.path.join(root, n))
print(json.dumps(results, indent=1))
from google.colab import files
files.download("ask-v6.zip")
print("Done: send ask-v6.zip. Runtime → Disconnect and delete runtime stops the units.")